# Saving Movie Files Example (simulation only)

Ported from: k-Wave/examples/example_ivp_saving_movie_files.m

Simulates the pressure field generated by two initial pressure discs in a
heterogeneous medium (spatially varying sound speed and density) detected by a
Cartesian circular sensor. Movie saving is skipped; only the simulation and
p_final recording are ported.

In [ ]:
!pip install k-wave-python

In [ ]:
import numpy as np

from kwave.data import Vector
from kwave.kgrid import kWaveGrid
from kwave.kmedium import kWaveMedium
from kwave.ksensor import kSensor
from kwave.ksource import kSource
from kwave.kspaceFirstOrder import kspaceFirstOrder
from kwave.utils.mapgen import make_cart_circle, make_disc

In [ ]:
def setup():
    """Set up the simulation physics (grid, medium, source).

    Returns:
        tuple: (kgrid, medium, source)
    """

    # create the computational grid
    Nx = 128
    Ny = 128
    dx = 0.1e-3  # [m]
    dy = 0.1e-3  # [m]
    kgrid = kWaveGrid(Vector([Nx, Ny]), Vector([dx, dy]))

    # define the properties of the propagation medium (heterogeneous)
    sound_speed = 1500 * np.ones((Nx, Ny))  # [m/s]
    sound_speed[: Nx // 2, :] = 1800  # [m/s]
    density = 1000 * np.ones((Nx, Ny))  # [kg/m^3]
    density[:, Ny // 4 :] = 1200  # [kg/m^3]

    medium = kWaveMedium(sound_speed=sound_speed, density=density)

    # create initial pressure distribution using make_disc
    # disc 1 (1-based positions for make_disc)
    disc_1 = 5 * make_disc(Vector([Nx, Ny]), Vector([50, 50]), 8)

    # disc 2
    disc_2 = 3 * make_disc(Vector([Nx, Ny]), Vector([80, 60]), 5)

    source = kSource()
    source.p0 = (disc_1 + disc_2).astype(float)

    # set time array
    kgrid.makeTime(sound_speed)

    return kgrid, medium, source

In [ ]:
def run(backend="python", device="cpu", quiet=True):
    """Run the simulation with a Cartesian circular sensor.

    Returns:
        dict: Simulation results with keys 'p' and 'p_final'.
    """
    kgrid, medium, source = setup()

    # define a centered circular sensor (Cartesian)
    sensor_radius = 4e-3  # [m]
    num_sensor_points = 50
    sensor_mask = make_cart_circle(sensor_radius, num_sensor_points)
    sensor = kSensor(mask=sensor_mask)
    sensor.record = ["p", "p_final"]

    return kspaceFirstOrder(
        kgrid,
        medium,
        source,
        sensor,
        backend=backend,
        device=device,
        quiet=quiet,
        pml_inside=True,
    )

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    result = run(quiet=False)
    p = np.asarray(result["p"])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # plot the sensor data
    ax = axes[0]
    im = ax.imshow(p, aspect="auto", cmap="RdBu_r")
    ax.set_ylabel("Sensor Position")
    ax.set_xlabel("Time Step")
    ax.set_title("Sensor Data")
    fig.colorbar(im, ax=ax)

    # plot the initial pressure with medium overlay
    kgrid, medium, source = setup()
    ax = axes[1]
    ax.imshow(
        source.p0.T,
        extent=[
            kgrid.x_vec[0] * 1e3,
            kgrid.x_vec[-1] * 1e3,
            kgrid.y_vec[-1] * 1e3,
            kgrid.y_vec[0] * 1e3,
        ],
        vmin=-5,
        vmax=5,
        cmap="RdBu_r",
    )
    ax.set_xlabel("y-position [mm]")
    ax.set_ylabel("x-position [mm]")
    ax.set_title("Initial Pressure Distribution")
    ax.set_aspect("equal")

    fig.suptitle("Saving Movie Files Example (simulation only)")
    fig.tight_layout()
    plt.show()